In [1]:
import pandas as pd

In [2]:
area_dict = {
    "Drammensfjord": ["011.", "012."],
    "Frierfjorden": ["016.A11", "016.41", "016.4A0", "016.42", "016.5", "016.32"],
    "Bunnefjorden": [
        "005.22",
        "005.311",
        "005.31Z",
        "005.312",
        "005.3A",
        "005.40",
        "005.4A",
        "006.11",
        "006.1A",
        "006.12",
        "006.21",
        "006.2A0",
        "006.22",
        "006.A10",
        "006.31",
        "006.3Z",
        "006.32",
    ],
    "Inner Oslofjord": ["005.", "006.", "007.", "008.", "009."],
}
pars = ["totn", "din", "ton", "totp", "tdp", "tpp", "toc", "ss"]
scens = ["Baseline", "Scenario_A", "Scenario_B"]
back_srcs = ["agriculture-background", "glacier", "upland", "wood", "lake"]

In [3]:
csv_path = r"/home/jovyan/shared/common/oslofjord_modelling/phase3_scenarios/teotil3_oslomod_results_2017-2019.csv"
df = pd.read_csv(csv_path)
df.head()

,scenario,year,regine,regine_down,accum_agriculture-background_din_kg,accum_agriculture-background_ss_kg,accum_agriculture-background_tdp_kg,accum_agriculture-background_toc_kg,accum_agriculture-background_ton_kg,accum_agriculture-background_totn_kg,...,local_urban_totp_kg,local_urban_tpp_kg,local_wood_din_kg,local_wood_ss_kg,local_wood_tdp_kg,local_wood_toc_kg,local_wood_ton_kg,local_wood_totn_kg,local_wood_totp_kg,local_wood_tpp_kg
0,Baseline,2017,001.10,001.,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,28.1,704.8,0.5,7753.1,145.2,173.3,4.3,3.8
1,Baseline,2017,001.1A2B,001.1A2A,101.705063,66.420112,0.444083,4936.322742,56.349583,158.054645,...,10.8,4.3,1026.0,26580.2,18.6,299026.9,5556.6,6582.6,163.5,144.9
2,Baseline,2017,001.1A4D,001.1A4C,13.761862,12.543802,0.024426,362.237626,5.818261,19.580123,...,0.0,0.0,184.9,4808.5,3.4,54576.8,1010.5,1195.4,29.8,26.4
3,Baseline,2017,001.1M,001.1L,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,368.5,9749.3,6.8,111142.1,2056.1,2424.6,60.9,54.1
4,Baseline,2017,001.21,001.,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,3.9,97.6,0.1,1078.3,20.2,24.1,0.6,0.5


In [4]:
for area, vassom_list in area_dict.items():
    print(area)
    for scen in scens:
        print(f"  {scen}")
        area_df = df.query("(scenario == @scen) and (regine in @vassom_list)").copy()
        area_df["vol_m3"] = area_df["accum_q_m3/s"] * 60 * 60 * 24 * 365.25
        for par in pars:
            cols = ["regine", "vol_m3"] + [
                col
                for col in area_df.columns
                if (col.startswith("accum_") and f"_{par}_" in col)
            ]
            par_df = area_df[cols].copy()
            par_df.columns = [col.replace("accum_", "") for col in par_df.columns]

            val_cols = [col for col in par_df.columns if col.endswith("_kg")]
            par_df[f"all_{par}_kg"] = par_df[val_cols].sum(axis="columns")
            nat_cols = [
                col for col in par_df.columns if col.startswith(tuple(back_srcs))
            ]
            par_df[f"natural_{par}_kg"] = par_df[nat_cols].sum(axis="columns")

            par_df = par_df.sum()
            if par.endswith(("n", "p")):
                # Calc ug/l
                scen_conc = 1e9 * par_df[f"all_{par}_kg"] / (1000 * par_df["vol_m3"])
                ref_conc = 1e9 * par_df[f"natural_{par}_kg"] / (1000 * par_df["vol_m3"])
            else:
                # mg/l
                scen_conc = 1e6 * par_df[f"all_{par}_kg"] / (1000 * par_df["vol_m3"])
                ref_conc = 1e6 * par_df[f"natural_{par}_kg"] / (1000 * par_df["vol_m3"])

            print(f"    {par}: {scen_conc:.1f} (reference {ref_conc:.1f})")

Drammensfjord
  Baseline
    totn: 474.4 (reference 179.3)
    din: 274.6 (reference 50.2)
    ton: 199.8 (reference 129.1)
    totp: 10.1 (reference 1.7)
    tdp: 4.1 (reference 1.0)
    tpp: 6.1 (reference 0.7)
    toc: 2.9 (reference 2.6)
    ss: 3.4 (reference 0.3)
  Scenario_A
    totn: 425.1 (reference 179.1)
    din: 235.2 (reference 50.0)
    ton: 189.9 (reference 129.0)
    totp: 7.9 (reference 1.7)
    tdp: 3.8 (reference 1.0)
    tpp: 4.1 (reference 0.7)
    toc: 2.8 (reference 2.6)
    ss: 1.6 (reference 0.3)
  Scenario_B
    totn: 374.4 (reference 175.0)
    din: 195.7 (reference 46.9)
    ton: 178.7 (reference 128.1)
    totp: 6.0 (reference 1.7)
    tdp: 3.0 (reference 1.0)
    tpp: 3.1 (reference 0.7)
    toc: 2.8 (reference 2.6)
    ss: 1.1 (reference 0.3)
Frierfjorden
  Baseline
    totn: 343.8 (reference 146.6)
    din: 206.4 (reference 43.8)
    ton: 137.5 (reference 102.8)
    totp: 5.0 (reference 1.0)
    tdp: 2.7 (reference 0.7)
    tpp: 2.3 (reference 0.2)
    t